# Gatekeeper, giai đoạn 0 (§37), phần A, **chạy theo lô**

Chạy bằng **Save Version → Save & Run All (Commit)**, không chạy từng ô. Kaggle chạy hết notebook trên máy riêng
(không cần mở trình duyệt, không bị tắt vì không ai thao tác) và giữ lại các file trong `/kaggle/working`.

Trước khi bấm: khung Settings bên phải **Internet: On**, **Accelerator: None**. Khoảng 40–60 phút.
Khi xong: mở phiên bản vừa chạy → tab **Output** → tải `k0a.json`, `k0a.log`, `k0b_kaggle.jsonl`, `k0b.log`, `k0_summary.json`.

In [ ]:
# 1 — cài đặt vào đúng Python của notebook
%pip -q install "swesmith==0.0.9" "swebench==4.1.0" udocker unidiff
import os  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = "/tmp/udocker"  # outside the output folder: not saved with the version
UD = "import sys; from udocker.maincmd import main; sys.argv[0] = 'udocker'; sys.exit(main())"
r = subprocess.run(
    [sys.executable, "-c", UD, "--allow-root", "install"], capture_output=True, text=True
)
print("udocker install:", r.returncode, (r.stdout + r.stderr).strip()[-200:])

In [ ]:
# 2 — mã và dữ liệu (repo công khai)
REPO = "dinhthithuthuyduphong-bot/gatekeeper"
BRANCH = "claude/gatekeeper-13-research-nxytoh"
os.chdir("/tmp")
if not os.path.isdir("/tmp/gatekeeper"):
    url = f"https://github.com/{REPO}"
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url], check=True)
CODE = "/tmp/gatekeeper/kaggle/scripts"
head = subprocess.run(
    ["git", "-C", "/tmp/gatekeeper", "log", "--oneline", "-1"], capture_output=True, text=True
)
print("code:", head.stdout.strip())
RAW = f"https://raw.githubusercontent.com/{REPO}"
D = "/tmp/k0data"
os.makedirs(D, exist_ok=True)
files = {
    "swesmith_pallets__jinja.jsonl": "data-snapshots/data/instances/swesmith_pallets__jinja.jsonl",
    "f1_probe.json": "trial/results/f1_probe.json",
    "f1_skill_probe.json": "trial/results/f1_skill_probe.json",
}
for name, path in files.items():
    subprocess.run(["curl", "-sSfL", "-o", f"{D}/{name}", f"{RAW}/{path}"], check=True)
    print(name, os.path.getsize(f"{D}/{name}"), "byte")

In [ ]:
# 3 — K0a (5–10 phút), chạy tiền cảnh; log lưu ở /kaggle/working/k0a.log
def run(cmd, log, cwd=None):
    with open(log, "w") as fh:
        p = subprocess.Popen(
            cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        for line in p.stdout:
            fh.write(line)
            fh.flush()
            print(line, end="", flush=True)
    rc = p.wait()
    print("exit code:", rc)
    return rc


IMAGE = "swebench/swesmith.x86_64.pallets_1776_jinja.ada0a9a6"
rc = run([sys.executable, f"{CODE}/k0a_probe.py", "--image", IMAGE, "--out-dir", W], f"{W}/k0a.log")
MODE = open(f"{W}/k0a_mode.txt").read().strip() if os.path.exists(f"{W}/k0a_mode.txt") else ""
print("==> mode:", MODE or "(none)")

In [ ]:
# 4 — K0b (30–45 phút), chạy tiền cảnh; kết quả lưu ở /kaggle/working/k0b_kaggle.jsonl
if MODE:
    os.environ["UDOCKER_EXECMODE"] = MODE
    os.makedirs("/tmp/k0run", exist_ok=True)
    args = [sys.executable, f"{CODE}/k0_stage0.py", "--rows", f"{D}/swesmith_pallets__jinja.jsonl"]
    args += ["--results", f"{D}/f1_probe.json", f"{D}/f1_skill_probe.json"]
    args += ["--controls", "20", "--out", f"{W}/k0b_kaggle.jsonl"]
    run(args, f"{W}/k0b.log", cwd="/tmp/k0run")
else:
    print("K0a found no execution mode: K0b not run")

In [ ]:
# 5 — tóm tắt (cũng lưu vào /kaggle/working/k0_summary.json)
import json  # noqa: E402

out = {"mode": MODE, "head": head.stdout.strip()}
if os.path.exists(f"{W}/k0b_kaggle.jsonl"):
    s = subprocess.run(
        [
            sys.executable,
            f"{CODE}/k0_stage0.py",
            "--rows",
            f"{D}/swesmith_pallets__jinja.jsonl",
            "--results",
            f"{D}/f1_probe.json",
            "--controls",
            "0",
            "--out",
            f"{W}/k0b_kaggle.jsonl",
            "--summary-only",
        ],
        capture_output=True,
        text=True,
    )
    out["K0b"] = json.loads(s.stdout)
json.dump(out, open(f"{W}/k0_summary.json", "w"), indent=1)
print(json.dumps(out, indent=1))
print(sorted(os.listdir(W)))